# Project 1: Create a simple Podacast

Authors: Akansha Verma & Louise Plessis

In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel
from scrapper import load_from_url, save_to_file
import gradio as gr
from pathlib import Path
#from typing import List

# Initialize client
from dotenv import load_dotenv
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # reads .env file in current directory
api_key = os.getenv("OPENAI_API_KEY")
print("Key loaded:", bool(api_key))

Key loaded: True


In [ ]:
INPUT_DATA="input_data/mindfulambition-net-beginners-mind.txt"
with open(INPUT_DATA, "rb") as input_file:
    content = input_file.read()
    print(f"Extracted text: {input_file}")

Extracted text: <_io.BufferedReader name='input_data/mindfulambition-net-beginners-mind.txt'>


## Process sources

In [ ]:
def get_summary(text_chunk):
    """Sends a single chunk to the LLM for summarization."""
    response = client.chat.completions.create(
        model="gpt-4o-mini", 
        messages=[
            {"role": "system", "content": "You are the witty, warm solo host of a podcast called 'The Joy of Learning.' Your job is to turn source material into an engaging spoken-word script — not a summary, a script meant to be read aloud by a text-to-speech voice.\n\nRules:\n1. Open with exactly this greeting style: 'Welcome to The Joy of Learning, the podcast where we explore [general subject in your own words].' Then transition naturally into the episode.\n2. Explain the core concept in your own words — do not summarize the source paragraph by paragraph, and do not reference the article, its author, or where the content came from. Write as if this is your own idea you're excited to share.\n3. Be genuinely funny: use a playful analogy, a light joke, or a self-aware aside at least once. Think 'smart friend explaining something cool over coffee,' not 'lecture.'\n4. Write only what should be spoken aloud — no headers, no stage directions, no markdown, no sound-effect cues.\n5. Keep it under 300 words so the TTS output stays a reasonable length.\n6. End with a short, warm sign-off that invites the listener to try adopting the mindset themselves."},
            {"role": "user", "content": f"Here is the source material to base the episode on:\n\n{text_chunk}"}
        ],
        temperature=0.8
    )
    print(response)
    return response.choices[0].message.content

res = get_summary(INPUT_DATA)  
print(res)


NameError: name 'INPUT_DATA' is not defined

## Generate Audio

In [ ]:
def text_to_speech_ai(text, output_file="audio_output.mp3"):
    response = client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    )
    
    # Save the binary audio stream to a file
    with open(output_file, "wb") as f:
        f.write(response.content)
    print(f"High-quality audio saved to {output_file}")

text_to_speech_ai(res)

High-quality audio saved to audio_output.mp3


## Generate Metata data (title & topic from the content)

In [ ]:
class EpisodeMeta(BaseModel):
    title: str
    topic: str

def generate_episode_metadata(source_text: str) -> EpisodeMeta:
    """Derive a catchy episode title + one-sentence topic from the source content."""
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "You generate metadata for episodes of a podcast called "
                "'The Joy of Learning.' Given source material, return:\n"
                "- title: a catchy, specific episode title (max 8 words), "
                "in the style of 'The Joy of Learning: <hook>'\n"
                "- topic: one sentence describing what the episode covers, "
                "written as if listing the themes/topics for a producer."
            )},
            {"role": "user", "content": f"Source material:\n\n{source_text[:6000]}"}
        ],
        response_format=EpisodeMeta,
    )
    return response.choices[0].message.parsed

In [ ]:
VOICES = ["alloy", "echo", "fable", "onyx", "nova", "shimmer"]

def generate_podcast(
    sources_text: str
):
    sources = [s.strip() for s in sources_text.splitlines() if s.strip()]
    if not sources:
        print("Please enter at least one source (URL or file path).")

    log_lines = []

    def log(msg):
        log_lines.append(msg)
        return "\n".join(log_lines)

    # 1. Load every source
    docs = []
    for i, src in enumerate(sources):
        try:
            doc = load_from_url(src)
            save_to_file(doc)
            docs.append(doc)
            log(f"✓ Loaded ({doc.source_type}): {src}  [{len(doc.raw_text.split())} words]")
        except Exception as e:
            log(f"✗ Failed to load {src}: {e}")
        #progress((i + 1) / len(sources) * 0.3, desc=f"Loaded {i+1}/{len(sources)} sources")

    if not docs:
        print("None of the sources could be loaded. Check the log for details.")

#progress(0.3, desc="Condensing sources...")
    combined_text = "\n\n".join(doc.raw_text for doc in docs)

    try:
        meta = generate_episode_metadata(combined_text)
        episode_title = meta.title
        log(f"✓ Generated title: {episode_title}")
        log(f"✓ Topic: {meta.topic}")
    except Exception as e:
        episode_title = "The Joy of Learning: Episode"
        log(f"✗ Metadata generation failed, using fallback: {e}")

    try:
        script = get_summary(combined_text)
    except Exception as e:
        print(f"Script generation failed: {e}")

    # 3. Text-to-speech
    #progress(0.75, desc="Generating audio (this can take a minute)...")
    safe_title = "".join(c if c.isalnum() or c in " -_" else "" for c in (episode_title or "episode"))
    output_path = f"output/{safe_title or 'episode'}.mp3"
    try:
        audio_path = text_to_speech_ai(script, output_file=output_path)
    except Exception as e:
        print(f"Audio generation failed: {e}")

    #progress(1.0, desc="Done!")
    return episode_title, script, audio_path, "\n".join(log_lines)


theme = gr.themes.Soft(
    primary_hue="orange",
    secondary_hue="yellow",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Poppins"), "ui-sans-serif", "sans-serif"],
)

custom_css = """
#header-banner {
    background: linear-gradient(135deg, #ff9a3c 0%, #ffcb52 100%);
    padding: 24px;
    border-radius: 16px;
    text-align: center;
    color: white;
    margin-bottom: 16px;
}
#header-banner h1 { font-size: 2.2em; margin: 0; }
#header-banner p { opacity: 0.9; margin-top: 4px; }
.generate-btn { font-size: 1.1em !important; }
"""

with gr.Blocks(title="The Joy of Learning — Podcast Studio", theme=theme, css=custom_css) as demo:
    gr.HTML(
        """
        <div id="header-banner">
            <h1>🎙️ The Joy of Learning</h1>
            <p>Turn articles & videos into a warm, witty podcast episode — automatically.</p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            sources_input = gr.Textbox(
                label="📥 Sources (one per line — article or YouTube URLs)",
                lines=8,
                placeholder="https://example.com/article\nhttps://youtube.com/watch?v=...",
            )
            voice_input = gr.Dropdown(VOICES, value="alloy", label="🎤 Voice")
            generate_btn = gr.Button("✨ Generate Podcast", variant="primary", elem_classes="generate-btn")

        with gr.Column(scale=1):
            title_output = gr.Textbox(label="📝 Generated Episode Title", interactive=False)
            audio_output = gr.Audio(label="🔊 Episode Audio", type="filepath")
            script_output = gr.Textbox(label="Episode Script", lines=12)
            log_output = gr.Textbox(label="Pipeline Log", lines=6)

    generate_btn.click(
        fn=generate_podcast,
        inputs=[sources_input],
        outputs=[title_output, script_output, audio_output, log_output],
    )

if __name__ == "__main__":
    Path("output").mkdir(exist_ok=True)
    demo.launch()

/var/folders/7d/mw8sk41d1rb_hx7v63zn3v180000gn/T/ipykernel_37203/2438363447.py:82: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="The Joy of Learning — Podcast Studio", theme=theme, css=custom_css) as demo:


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
